# Cleaning a Messy Clinical-Style Dataset

**Author:** Bilal Ahmad — Healthcare Data Analyst

> **Note:** This notebook is for learning and portfolio demonstration only. It is **not** clinical or medical advice, and it uses **no real patient data**. **SYNTHETIC DEMO DATA — NOT REAL PATIENTS.** Every row below was invented inside this notebook to demonstrate cleaning steps. No real patient record is used anywhere in this notebook.


## 1. Introduction

Real-world health data is rarely clean. This notebook creates a small, deliberately messy synthetic dataset and fixes it step by step: missing values, exact duplicates, inconsistent category labels, and impossible outliers. Every step is checked with a before/after comparison, ending in a short QA summary.

In [1]:
import numpy as np
import pandas as pd

# SYNTHETIC DEMO DATA — invented for teaching; not real patients.
raw = pd.DataFrame({
    "patient_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 11, 12, 13, 14, 15],
    "age": [45, 62, np.nan, 34, 250, 58, 41, np.nan, 73, 29, 29, 51, 38, 66, 47, 55],
    "gender": ["Male", "female", "M", "Female", "male", "F", "Male", "Female",
               "female", "Male", "Male", "F", "M", "Female", "male", "Female"],
    "diagnosis": ["Diabetes", "hypertension", "Diabetes", "DIABETES", "Hypertension",
                  "diabetes", "Asthma", "asthma", "Hypertension", "Diabetes",
                  "Diabetes", "ASTHMA", "hypertension", "Diabetes", "Asthma", "diabetes"],
    "systolic_bp": [130, 145, 118, np.nan, 320, 152, 121, 138, 160, 115, 115, 142, np.nan, 155, 128, 133],
})
print("Raw shape:", raw.shape)
print("\nMissing values per column:")
print(raw.isna().sum())
print("\nExact duplicate rows:", int(raw.duplicated().sum()))
print("\nUnique gender labels:", sorted(raw["gender"].unique()))
print("Unique diagnosis labels:", sorted(raw["diagnosis"].unique()))
print("\nAge / systolic_bp extremes (before cleaning):")
print(raw[["age", "systolic_bp"]].describe().T[["min", "max"]])

Raw shape: (16, 5)

Missing values per column:
patient_id     0
age            2
gender         0
diagnosis      0
systolic_bp    2
dtype: int64

Exact duplicate rows: 1

Unique gender labels: ['F', 'Female', 'M', 'Male', 'female', 'male']
Unique diagnosis labels: ['ASTHMA', 'Asthma', 'DIABETES', 'Diabetes', 'Hypertension', 'asthma', 'diabetes', 'hypertension']

Age / systolic_bp extremes (before cleaning):
               min    max
age           29.0  250.0
systolic_bp  115.0  320.0


## 2. Step 1 — Remove Exact Duplicates

Duplicate rows double-count a patient and distort every average. They are removed first, keeping the first occurrence.

In [2]:
df = raw.drop_duplicates().reset_index(drop=True)
print("Shape after de-duplication:", df.shape)

Shape after de-duplication: (15, 5)


## 3. Step 2 — Standardise Category Labels

The same category was typed several ways ('Male', 'male', 'M' / 'Diabetes', 'DIABETES', 'diabetes'). Standardising to one label per category prevents a single group being split into several in later analysis.

In [3]:
df["gender"] = df["gender"].str.strip().str.lower().map({"male": "Male", "m": "Male", "female": "Female", "f": "Female"})
df["diagnosis"] = df["diagnosis"].str.strip().str.lower().str.title()
print("Gender labels now:", sorted(df["gender"].unique()))
print("Diagnosis labels now:", sorted(df["diagnosis"].unique()))

Gender labels now: ['Female', 'Male']
Diagnosis labels now: ['Asthma', 'Diabetes', 'Hypertension']


## 4. Step 3 — Handle Missing Values

Numeric gaps are filled with the **median** (not the mean), because the median is unaffected by the extreme values still present at this stage. The counts are reported so the reader knows how much was imputed.

In [4]:
for col in ["age", "systolic_bp"]:
    n_missing = int(df[col].isna().sum())
    df[col] = df[col].fillna(df[col].median())
    print(f"{col}: filled {n_missing} missing value(s) with median {df[col].median():.1f}")
print("\nMissing values remaining:", int(df.isna().sum().sum()))

age: filled 2 missing value(s) with median 51.0
systolic_bp: filled 2 missing value(s) with median 138.0

Missing values remaining: 0


## 5. Step 4 — Handle Impossible Outliers

Ages above 110 and systolic blood pressures above 250 are not plausible in this demo data, so they are clipped to those plausible limits and flagged in the report. In real work, such values would be checked against the source record first — never silently deleted.

In [5]:
df["age"] = df["age"].clip(upper=110)
df["systolic_bp"] = df["systolic_bp"].clip(upper=250)
print(df[["age", "systolic_bp"]].describe().T[["min", "max"]].round(1))

               min    max
age           29.0  110.0
systolic_bp  115.0  250.0


## 6. Before / After QA Summary

A cleaning step is only finished when it can be verified. The table below compares the dataset before and after cleaning.

In [6]:
qa = pd.DataFrame({
    "Check": ["Rows", "Missing values (total)", "Exact duplicate rows",
              "Distinct gender labels", "Distinct diagnosis labels",
              "Max age", "Max systolic_bp"],
    "Before": [len(raw), int(raw.isna().sum().sum()), int(raw.duplicated().sum()),
               raw["gender"].nunique(), raw["diagnosis"].nunique(),
               raw["age"].max(), raw["systolic_bp"].max()],
    "After": [len(df), int(df.isna().sum().sum()), int(df.duplicated().sum()),
              df["gender"].nunique(), df["diagnosis"].nunique(),
              df["age"].max(), df["systolic_bp"].max()],
})
print(qa.to_string(index=False))
print("\nCleaned dataset preview:")
print(df.head(10).to_string(index=False))

                    Check  Before  After
                     Rows    16.0   15.0
   Missing values (total)     4.0    0.0
     Exact duplicate rows     1.0    0.0
   Distinct gender labels     6.0    2.0
Distinct diagnosis labels     8.0    3.0
                  Max age   250.0  110.0
          Max systolic_bp   320.0  250.0

Cleaned dataset preview:
 patient_id   age gender    diagnosis  systolic_bp
          1  45.0   Male     Diabetes        130.0
          2  62.0 Female Hypertension        145.0
          3  51.0   Male     Diabetes        118.0
          4  34.0 Female     Diabetes        138.0
          5 110.0   Male Hypertension        250.0
          6  58.0 Female     Diabetes        152.0
          7  41.0   Male       Asthma        121.0
          8  51.0 Female       Asthma        138.0
          9  73.0 Female Hypertension        160.0
         10  29.0   Male     Diabetes        115.0


## 7. Conclusion

The cleaned demo dataset is now ready for analysis: one row per patient, one label per category, no missing values, and no impossible extremes. The key habits to carry into real work are: (1) measure the mess before fixing it, (2) prefer reversible, documented fixes (median imputation, clipping with limits stated), and (3) always finish with a before/after QA check. Note that this demo is tiny on purpose — with real clinical data, each fix would also be validated against the source system and documented in a data dictionary.